In [1]:
# TabDDPM vendor checkout lives at SYNTH_BENCHMARK/_vendor/tab-ddpm (see README).
# From the repository root, run once:
#   git clone https://github.com/yandex-research/tab-ddpm _vendor/tab-ddpm

In [ ]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
# patten-server GPU: pip install torch --index-url https://download.pytorch.org/whl/cu124
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
DIFFUSION_PKG = NOTEBOOK_DIR.parent
_repo = NOTEBOOK_DIR
while not (_repo / "_vendor" / "tab-ddpm").is_dir() and _repo.parent != _repo:
    _repo = _repo.parent
REPO_ROOT = _repo
if not (REPO_ROOT / "_vendor" / "tab-ddpm").is_dir():
    raise FileNotFoundError(
        "Missing _vendor/tab-ddpm. From SYNTH_BENCHMARK root run: "
        "git clone https://github.com/yandex-research/tab-ddpm _vendor/tab-ddpm"
    )
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import (
    train_tabddpm,
    train_forestdiffusion,
    resolve_experiment_device,
    print_experiment_runtime,
)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality


def normalize_eval_frame(df, label_col, reference=None):
    """Cast labels to int when the reference column is numeric."""
    out = df.copy()
    ref = pd.Series(reference if reference is not None else out[label_col])
    if pd.to_numeric(ref, errors="coerce").notna().all():
        out[label_col] = pd.to_numeric(out[label_col], errors="coerce").round().astype(int)
    else:
        out[label_col] = out[label_col].astype(str)
    return out


def require_experiment_context(*extra_vars: str) -> None:
    """Ensure experiment settings were run before generator cells."""
    required = [
        "GENERATORS_TO_EVAL",
        "train_diffusion",
        "train_trtr",
        "test_gen",
        "target_col",
        "N_DIFFUSION_SYNTH_SAMPLES",
        "train_diffusion_metadata",
        "synthetic_datasets",
        "scores",
        "RUN_QUALITY_EVAL",
        "SEED",
        "EXPERIMENT_DEVICE",
    ] + list(extra_vars)
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Experiment context is not initialized. Missing: "
            + ", ".join(missing)
            + ". Run the Experiment Settings cell first."
        )



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
covertype = fetch_ucirepo(id=31)

# data (as pandas dataframes)
X = covertype.data.features
y = covertype.data.targets

# metadata
print(covertype.metadata)

# variable information
print(covertype.variables)

covertype_data = pd.concat([X, y], axis=1)

# ----------------------------------------------------
# Target column handling
# ----------------------------------------------------
# Ensure Cover_Type is the explicit target variable
if "Cover_Type" in covertype_data.columns:
    target_col = "Cover_Type"
elif "cover_type" in covertype_data.columns:
    covertype_data = covertype_data.rename(columns={"cover_type": "Cover_Type"})
    target_col = "Cover_Type"
else:
    target_col = covertype_data.columns[-1]
    covertype_data = covertype_data.rename(columns={target_col: "Cover_Type"})
    target_col = "Cover_Type"

# Keep full dataset for repeatable subsampling when experiment settings is re-run.
covertype_data_full = covertype_data.copy()
print(f"Full dataset cached: {covertype_data_full.shape}")


{'uci_id': 31, 'name': 'Covertype', 'repository_url': 'https://archive.ics.uci.edu/dataset/31/covertype', 'data_url': 'https://archive.ics.uci.edu/static/public/31/data.csv', 'abstract': 'Classification of pixels into 7 forest cover types based on attributes such as elevation, aspect, slope, hillshade, soil-type, and more.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 581012, 'num_features': 54, 'feature_types': ['Categorical', 'Integer'], 'demographics': [], 'target_col': ['Cover_Type'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1998, 'last_updated': 'Sat Mar 16 2024', 'dataset_doi': '10.24432/C50K5N', 'creators': ['Jock Blackard'], 'intro_paper': None, 'additional_info': {'summary': 'Predicting forest cover type from cartographic variables only (no remotely sensed data).  The actual forest cover type for a given observation (30 x 30 meter cell) was determined from

In [4]:
# ----------------------------------------------------
# Preprocess features before synthetic data generation
# ----------------------------------------------------

X = covertype_data_full.drop(columns=[target_col]).copy()
y = covertype_data_full[target_col].copy()

covertype_data_full = pd.concat([X, y], axis=1)
covertype_data = covertype_data_full.copy()

print(f'Target variable: {target_col}')
print(f'Dataset shape after preprocessing: {covertype_data_full.shape}')


Target variable: Cover_Type
Dataset shape after preprocessing: (581012, 55)


In [5]:
# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000          # random real samples drawn from full dataset
TEST_SIZE = 0.2           # 20% holdout for unseen TSTR evaluation
SEED = 42

# PyTorch device for TabDDPM: "auto" | "cuda" | "cuda:0" | "cpu"
DEVICE = "auto"
EXPERIMENT_DEVICE = resolve_experiment_device(DEVICE)

# Speed controls (set DEV_MODE=False, FAST_MODE=False for full paper run)
FAST_MODE = True
DEV_MODE = True
RUN_QUALITY_EVAL = True

# Diffusion generators: 1000 stratified real rows -> 1000 synthetic rows
N_DIFFUSION_SAMPLES = 1000
N_DIFFUSION_SYNTH_SAMPLES = 1000

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = ["ForestDiffusion", "TabDDPM"]
GENERATORS_TO_EVAL = ALL_GENERATORS

# Diffusion models use these 10 cartographic features + Cover_Type target.
CONTINUOUS_COLS = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]

GENERATOR_FEATURE_COLS = CONTINUOUS_COLS.copy()
if len(covertype_data_full) < N_SAMPLES:
    raise RuntimeError(
        f"Full dataset has only {len(covertype_data_full)} rows. "
        "Re-run the data loading and preprocessing cells first."
    )

covertype_data, _ = train_test_split(
    covertype_data_full,
    train_size=N_SAMPLES,
    stratify=covertype_data_full[target_col],
    random_state=SEED,
)
covertype_data = covertype_data.reset_index(drop=True)

# 80% for generator training, 20% held out unseen for TSTR evaluation.
train_real, test_real = train_test_split(
    covertype_data,
    test_size=TEST_SIZE,
    stratify=covertype_data[target_col],
    random_state=SEED,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)

GENERATOR_FEATURE_COLS = [c for c in GENERATOR_FEATURE_COLS if c in train_real.columns]
if len(GENERATOR_FEATURE_COLS) != 10:
    raise ValueError(
        f"Expected 10 shared generator features, found {len(GENERATOR_FEATURE_COLS)}: "
        f"{GENERATOR_FEATURE_COLS}"
    )

GEN_COLS = GENERATOR_FEATURE_COLS + [target_col]
train_trtr = train_real[GEN_COLS].copy()
test_gen = test_real[GEN_COLS].copy()

train_trtr[target_col] = pd.to_numeric(train_trtr[target_col], errors="coerce").round().astype(int)
test_gen[target_col] = pd.to_numeric(test_gen[target_col], errors="coerce").round().astype(int)

# Diffusion models train on the full stratified 1000-row subsample (not the 80/20 split).
if len(covertype_data) != N_DIFFUSION_SAMPLES:
    raise ValueError(
        f"Expected {N_DIFFUSION_SAMPLES} stratified rows for diffusion training, "
        f"found {len(covertype_data)}. Re-run the subsampling cell."
    )
train_diffusion = covertype_data[GEN_COLS].copy().reset_index(drop=True)
train_diffusion[target_col] = pd.to_numeric(
    train_diffusion[target_col], errors="coerce"
).round().astype(int)

train_diffusion_metadata = SingleTableMetadata()
train_diffusion_metadata.detect_from_dataframe(train_diffusion)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []

print(f'Random subsample: {covertype_data.shape}')
print(f'TRTR training set (80% of subsample): {train_trtr.shape}')
print(f'Diffusion training set (full stratified subsample): {train_diffusion.shape}')
print(f'Holdout test set: {test_gen.shape}')
print(f'DEV_MODE: {DEV_MODE} | FAST_MODE: {FAST_MODE} | quality eval: {RUN_QUALITY_EVAL}')
print(f'Diffusion generators: {GENERATORS_TO_EVAL}')
print(f'Diffusion synthetic samples: {N_DIFFUSION_SYNTH_SAMPLES}')
print(f'Features + target: {GEN_COLS}')
print_experiment_runtime(EXPERIMENT_DEVICE)


Random subsample: (1000, 55)
TRTR training set (80% of subsample): (800, 11)
Diffusion training set (full stratified subsample): (1000, 11)
Holdout test set: (200, 11)
DEV_MODE: True | FAST_MODE: True | quality eval: True
Diffusion generators: ['ForestDiffusion', 'TabDDPM']
Diffusion synthetic samples: 1000
Features + target: ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Cover_Type']
Experiment runtime
PyTorch version: 2.13.0+cu130
Selected device: cpu
CUDA available: False
TabDDPM will run on CPU.
ForestDiffusion: CPU (tree/flow model, n_jobs=-1)


In [6]:
# ---------------------------------------------------
# SINGLE RUN — setup + TabDDPM
# ---------------------------------------------------
require_experiment_context()
seed = SEED

print("\n================ SINGLE RUN ================")
print(f"TabDDPM device: {EXPERIMENT_DEVICE}")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if EXPERIMENT_DEVICE.startswith("cuda"):
    gpu_id = int(EXPERIMENT_DEVICE.split(":")[1]) if ":" in EXPERIMENT_DEVICE else 0
    torch.cuda.set_device(gpu_id)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.empty_cache()

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_diffusion,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_DIFFUSION_SYNTH_SAMPLES,
            seed=seed,
            device=EXPERIMENT_DEVICE,
            fast_mode=FAST_MODE,
        )
        synthetic_tabddpm = normalize_eval_frame(
            synthetic_tabddpm,
            target_col,
            reference=test_gen[target_col],
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_diffusion,
                synthetic_data=synthetic_tabddpm,
                metadata=train_diffusion_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')


================ SINGLE RUN ================
TabDDPM device: cpu
Training TabDDPM...
[0]
17
{'num_classes': 7, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(17)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.3128 Sum: 0.3128
Step 1000/1000 MLoss: 0.0 GLoss: 0.2829 Sum: 0.2829
mlp
Sample timestep    0
Discrete cols: []
Num shape:  (1000, 10)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 534.70it/s]|
Column Shapes Score: 73.66%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 271.07it/s]|
Column Pair Trends Score: 76.2%

Overall Score (Average): 74.93%

TabDDPM: 0.7493


In [7]:
# ForestDiffusion
require_experiment_context()
seed = SEED

if 'ForestDiffusion' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training ForestDiffusion...')
        print(f'ForestDiffusion fast_mode={FAST_MODE} (CPU/XGBoost, n_jobs capped)')
        synthetic_forestdiffusion = train_forestdiffusion(
            train_diffusion,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_DIFFUSION_SYNTH_SAMPLES,
            seed=seed,
            fast_mode=FAST_MODE,
        )
        synthetic_forestdiffusion = normalize_eval_frame(
            synthetic_forestdiffusion,
            target_col,
            reference=test_gen[target_col],
        )
        synthetic_datasets['ForestDiffusion'] = synthetic_forestdiffusion.copy()
        print('ForestDiffusion: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_diffusion,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_diffusion_metadata,
            )
            scores['ForestDiffusion'] = quality.get_score()
            print('ForestDiffusion:', round(scores['ForestDiffusion'], 4))
        else:
            print('ForestDiffusion: trained (quality eval skipped)')
    except Exception as e:
        print('ForestDiffusion Failed (training/sampling):')
        traceback.print_exc()
    if 'ForestDiffusion' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)')


Training ForestDiffusion...
ForestDiffusion device: CPU (n_jobs=-1)
ForestDiffusion: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 826.51it/s]|
Column Shapes Score: 94.68%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 313.56it/s]|
Column Pair Trends Score: 96.75%

Overall Score (Average): 95.71%

ForestDiffusion: 0.9571


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(kernel="rbf", cache_size=1000, tol=1e-3, random_state=42),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")

Classifier evaluation: 10 models, 10 seeds
FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.


In [9]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd


In [10]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


Skipping duplicate TRTR cell. Run the comparison cell for TRTR/TSTR (10 models, 10 seeds).


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def _normalize_labels(y, reference=None):
    """Align label dtypes so sklearn metrics do not mix strings and numbers."""
    y = pd.Series(y).reset_index(drop=True)
    ref = pd.Series(reference).reset_index(drop=True) if reference is not None else y
    ref_numeric = pd.to_numeric(ref, errors="coerce")
    if ref_numeric.notna().all():
        return pd.to_numeric(y, errors="coerce").round().astype(int)
    return y.astype(str)


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
    use_holdout=False,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    y_test_reference = _normalize_labels(test_df[label_col])

    def _std(values):
        return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = _normalize_labels(train_df[label_col], reference=y_test_reference)
            X_test = test_df.drop(columns=[label_col])
            y_test = y_test_reference.copy()

            if use_holdout:
                # Fixed holdout test; resample training rows per seed.
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )

                _, X_test, _, y_test = train_test_split(
                    X_test,
                    y_test,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_test),
                )

            clf = clone(model)
            if 'random_state' in clf.get_params():
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train, y_train)
            y_pred = _normalize_labels(clf.predict(X_test), reference=y_test_reference)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, average="weighted", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": _std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": _std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": _std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": _std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {_std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {_std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {_std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {_std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [12]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = "Cover_Type"

model_order = [
    "ForestDiffusion",
    "TabDDPM",
]

seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real) — 80% train / 20% holdout")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Diffusion generators: {len(model_order)}"
)
print(f"TRTR training set: {train_trtr.shape} | Holdout test set: {test_gen.shape}")

trtr_results = evaluate_models(
    train_df=train_trtr,
    test_df=test_gen,
    label_col="Cover_Type",
    models=models,
    seeds=seeds,
    use_holdout=True,
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR (train on synthetic, test on 20% holdout)")

    synthetic_train_df = normalize_eval_frame(
        synthetic_datasets[synth_name],
        label_col,
        reference=test_gen[label_col],
    )

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_gen,
        label_col="Cover_Type",
        models=models,
        seeds=seeds,
        use_holdout=True,
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)

TRTR (Train Real, Test Real) — 80% train / 20% holdout
Classifiers: 10 | Seeds: 10 | Diffusion generators: 2
TRTR training set: (800, 11) | Holdout test set: (200, 11)
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
7,GradientBoost,0.6915 ± 0.0180,0.6843 ± 0.0186,0.6880 ± 0.0221,0.6915 ± 0.0180
6,ExtraTrees,0.6825 ± 0.0189,0.6620 ± 0.0217,0.6587 ± 0.0342,0.6825 ± 0.0189
5,RandomForest,0.6815 ± 0.0190,0.6603 ± 0.0210,0.6583 ± 0.0332,0.6815 ± 0.0190
0,LogReg,0.6565 ± 0.0138,0.6313 ± 0.0142,0.6373 ± 0.0210,0.6565 ± 0.0138
8,AdaBoost,0.6485 ± 0.0278,0.6197 ± 0.0226,0.6169 ± 0.0259,0.6485 ± 0.0278
1,SVM-RBF,0.6410 ± 0.0156,0.6088 ± 0.0172,0.5935 ± 0.0169,0.6410 ± 0.0156
4,DecisionTree,0.6100 ± 0.0231,0.6033 ± 0.0236,0.6061 ± 0.0226,0.6100 ± 0.0231
2,KNN,0.6050 ± 0.0199,0.5859 ± 0.0203,0.5835 ± 0.0316,0.6050 ± 0.0199
3,NaiveBayes,0.5980 ± 0.0187,0.5955 ± 0.0154,0.5978 ± 0.0125,0.5980 ± 0.0187
9,MLP,0.4790 ± 0.0490,0.4506 ± 0.0420,0.5037 ± 0.0460,0.4790 ± 0.0490


ForestDiffusion - TSTR (train on synthetic, test on 20% holdout)
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,0.8000 ± 0.0189,0.7926 ± 0.0192,0.8061 ± 0.0185,0.8000 ± 0.0189
5,RandomForest,0.7780 ± 0.0123,0.7706 ± 0.0137,0.7799 ± 0.0152,0.7780 ± 0.0123
7,GradientBoost,0.7370 ± 0.0160,0.7319 ± 0.0147,0.7368 ± 0.0144,0.7370 ± 0.0160
4,DecisionTree,0.6940 ± 0.0307,0.6934 ± 0.0309,0.7002 ± 0.0305,0.6940 ± 0.0307
8,AdaBoost,0.6835 ± 0.0118,0.6519 ± 0.0146,0.6329 ± 0.0125,0.6835 ± 0.0118
2,KNN,0.6705 ± 0.0146,0.6577 ± 0.0149,0.6679 ± 0.0177,0.6705 ± 0.0146
0,LogReg,0.6670 ± 0.0144,0.6422 ± 0.0150,0.6596 ± 0.0145,0.6670 ± 0.0144
1,SVM-RBF,0.6545 ± 0.0119,0.6244 ± 0.0112,0.6172 ± 0.0100,0.6545 ± 0.0119
3,NaiveBayes,0.6465 ± 0.0113,0.6407 ± 0.0125,0.6369 ± 0.0145,0.6465 ± 0.0113
9,MLP,0.5170 ± 0.0471,0.5132 ± 0.0399,0.5567 ± 0.0315,0.5170 ± 0.0471


ForestDiffusion - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
3,NaiveBayes,0.4595 ± 0.0284,0.3578 ± 0.0295,0.3435 ± 0.0490,0.4595 ± 0.0284
8,AdaBoost,0.4565 ± 0.0379,0.3965 ± 0.0387,0.3775 ± 0.0336,0.4565 ± 0.0379
1,SVM-RBF,0.4520 ± 0.0136,0.3567 ± 0.0168,0.3471 ± 0.0192,0.4520 ± 0.0136
0,LogReg,0.4495 ± 0.0146,0.3578 ± 0.0166,0.3450 ± 0.0175,0.4495 ± 0.0146
6,ExtraTrees,0.4040 ± 0.0197,0.3669 ± 0.0208,0.3387 ± 0.0202,0.4040 ± 0.0197
2,KNN,0.4000 ± 0.0266,0.3749 ± 0.0228,0.3540 ± 0.0202,0.4000 ± 0.0266
7,GradientBoost,0.3970 ± 0.0204,0.3680 ± 0.0170,0.3446 ± 0.0157,0.3970 ± 0.0204
5,RandomForest,0.3945 ± 0.0383,0.3598 ± 0.0356,0.3325 ± 0.0333,0.3945 ± 0.0383
4,DecisionTree,0.3610 ± 0.0317,0.3582 ± 0.0292,0.3582 ± 0.0273,0.3610 ± 0.0317
9,MLP,0.3395 ± 0.0743,0.3178 ± 0.0588,0.3404 ± 0.0489,0.3395 ± 0.0743


TabDDPM - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,TabDDPM,GradientBoost,0.2945,0.316317,0.343457,0.2945,0.6915 ± 0.0180,0.3970 ± 0.0204
1,TabDDPM,ExtraTrees,0.2785,0.295109,0.319932,0.2785,0.6825 ± 0.0189,0.4040 ± 0.0197
2,TabDDPM,RandomForest,0.2870,0.300468,0.325808,0.2870,0.6815 ± 0.0190,0.3945 ± 0.0383
3,TabDDPM,LogReg,0.2070,0.273558,0.292328,0.2070,0.6565 ± 0.0138,0.4495 ± 0.0146
4,TabDDPM,AdaBoost,0.1920,0.223199,0.239381,0.1920,0.6485 ± 0.0278,0.4565 ± 0.0379
5,TabDDPM,SVM-RBF,0.1890,0.252145,0.246365,0.1890,0.6410 ± 0.0156,0.4520 ± 0.0136
6,TabDDPM,DecisionTree,0.2490,0.245126,0.247891,0.2490,0.6100 ± 0.0231,0.3610 ± 0.0317
7,TabDDPM,KNN,0.2050,0.211013,0.229485,0.2050,0.6050 ± 0.0199,0.4000 ± 0.0266
8,TabDDPM,NaiveBayes,0.1385,0.237721,0.254294,0.1385,0.5980 ± 0.0187,0.4595 ± 0.0284
9,TabDDPM,MLP,0.1395,0.132763,0.163321,0.1395,0.4790 ± 0.0490,0.3395 ± 0.0743


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,-0.05545,-0.061691,-0.065031,-0.05545
1,TabDDPM,0.21800,0.248742,0.266226,0.21800


In [13]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")


Results saved to: TRTR_TSTR_results.xlsx
